# **Goal**

The purpose of this notebook is to present some initial experiments to motivate model selection for an interactive app that allows users to experiment with small-medium language model (100M - 1B range).

###**Pythia Model Family**

- Pythia models were originally designed for interpretability research.

- All Pythia models were trained on the same data in the same order.

- Training data: All Pythia models are trained on The Pile. The Pile is a 825GiB general-purpose dataset in English. It contains texts from 22 diverse sources, roughly broken down into five categories: academic writing (e.g. arXiv), internet (e.g. CommonCrawl), prose (e.g. Project Gutenberg), dialogue (e.g. YouTube subtitles), and miscellaneous (e.g. GitHub, Enron Emails).

- Each model saw 299,892,736,000 ~= 300B tokens during training.

####**Pythia 410M**

- <u>Total params</u>: 405,334,016. This is the total number of learned scalar values in the entire model: weights, biases, embeddings, output projection, etc.
- <u>Non-embdedding params</u>: 302,311,424. These are the learned parameters in the transformer machinery itself, excluding the token embedding and output vocabulary matrices.
- <u>Number of layers</u>: 24
- <u>Model dimension</u>: 1024. I.e.: Every token is represented internally by a vector of length 1024.
- <u>MLP units per layer:</u> 4096
- <u>Heads</u>: 16
- <u>Vocabulary size:</u> 50,304. Amount of tokens in vocabulary.
- <u>Max context window</u>: 2048 tokens.

Load model

In [1]:
from transformers import GPTNeoXForCausalLM, AutoTokenizer

model = GPTNeoXForCausalLM.from_pretrained(
  "EleutherAI/pythia-410m-deduped",
  revision="step143000",
  cache_dir="./pythia-c-deduped/step143000",
)

tokenizer = AutoTokenizer.from_pretrained(
  "EleutherAI/pythia-410m-deduped",
  revision="step143000",
  cache_dir="./pythia-410m-deduped/step143000",
)

inputs = tokenizer("Hello, I am", return_tensors="pt")
tokens = model.generate(**inputs)
tokenizer.decode(tokens[0])

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.62GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1676: UserWarning: Using the model-agnostic default `max_length` (=24) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


'Hello, I am a newbie to the world of programming. I am trying to create a program that will take a'

Prompt experiments. Can play with the following generation parameters:

- `max_new_tokens:` Intenger determining length of response excluding initial prompt.
- `do_sample:` Boolean determining greedy/non-greedy sampling.
- `temperature:` Low temp -> sharp distribution; high temp -> flattened distribution. I.e., how predictable or conservative the model is when sampling tokens.
- `top_k:` Only consider the \(k\) highest-scoring tokens. Default = 50.
- `top_p:` Float 0-1.0. Keep the smallest set of most likely tokens whose probabilities together account for at least \(p\) of the probability mass. Default = 1.
- `repetition_penalty:` How much repeated tokens are discouraged. Default = 1.0 means no penalty.

In [2]:
inputs = tokenizer("How much do you like ice cream?", return_tensors="pt")
tokens = model.generate(**inputs)
tokenizer.decode(tokens[0])

/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1676: UserWarning: Using the model-agnostic default `max_length` (=28) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


'How much do you like ice cream?\n\nI like ice cream. I like it so much that I can’t stop eating it'

In [3]:
inputs = tokenizer("The capital of France is", return_tensors="pt")
tokens = model.generate(**inputs)
tokenizer.decode(tokens[0])

/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1676: UserWarning: Using the model-agnostic default `max_length` (=25) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


'The capital of France is the most populous city in the world, with a population of over 1.3 million. It'

In [4]:

inputs = tokenizer("The capital of France is", return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
)

print(tokenizer.decode(output[0]))

The capital of France is a former colony of the French Crown. Its capital is Paris.

Paris is the capital of France. It is the largest city in the country with a population of around 23 million inhabitants. In 2000, Paris became the capital of the French Republic


In [5]:
inputs = tokenizer("What is love?", return_tensors="pt")

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Input tokens: {input_tokens}")
print(f"Output tokens: {new_tokens}")
print()
print(tokenizer.decode(output[0], skip_special_tokens=True))

Input tokens: tensor([1276,  310, 2389,   32])
Output tokens: tensor([ 187,  187, 2598, 1142,  273,  253, 6281,  326,  309, 1849, 1239,  689,
         253, 1107,  403, 2032,   15, 1707,  434,  247,  629,  275,  619, 1481,
         835,  352, 2296,   13,  346,   42,  816,  971,  281,  320, 5211,  449,
        1244,  347, 2080,  896,  347,  309,  476, 4456,  285, 3164, 1078,  840,
        1051,  309])

What is love?

So many of the stories that I've read over the years are true. There's a part in my head where it says, "I just want to be happy." And as far back as I can remember and probably before then... I


Here we are just printing the internal shape of one layer of the model.

In [6]:
print(model.gpt_neox.layers[0].mlp)

GPTNeoXMLP(
  (dense_h_to_4h): Linear(in_features=1024, out_features=4096, bias=True)
  (dense_4h_to_h): Linear(in_features=4096, out_features=1024, bias=True)
  (act): GELUActivation()
)


In [7]:
model.config.intermediate_size

4096

The input vector representation is 1024-dimensional. The number of MLP units in the layer is 4096. Each of these produces a scalar from the input through learned weights and bises, so the layer produces a transformed and expanded 4096-d vector. After activation using GELU (Gaussian Error Linear Unit), these values are transformed back into a new 1024-d representation by another learned matrix. This representation will continue through the transformer block and subsequent layers.

Note: $\text{GELU}(x) = x \cdot \Phi(x) = 0.5x \left(1 + \text{erf}\left(\frac{x}{\sqrt{2}}\right)\right)$, where $Φ(x)$ is the cumulative distribution function of the standard normal distribution, and $\text{erf}(x)$ is the Gauss error function.

**MLP Unit Ablation Experiments**

In [8]:
import torch

def add_mlp_ablation(model, layer_idx, proportion, seed=42):

    # Load weights in layer layer_idx
    mlp = model.gpt_neox.layers[layer_idx].mlp

    num_units = model.config.intermediate_size
    num_ablate = int(proportion * num_units)

    # Seed RNG
    generator = torch.Generator()
    generator.manual_seed(seed)

    # Pick 'num_ablate' indexes for MLP units at random
    # Indexes in [0, 4095]
    indices = torch.randperm(
        num_units,
        generator=generator
    )[:num_ablate]

    #
    def ablation_hook(module, inputs, output):
        indices_device = indices.to(output.device)
        output = output.clone()
        output[..., indices_device] = 0
        return output

    handle = mlp.act.register_forward_hook(ablation_hook)

    return handle, indices, layer_idx

Here we try zeroing the outputs of different proportions of the first (0th) layer. Notice gradual degradation in output coherence.

Baseline

In [9]:
handle, indices, layer_id = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=0.0,
)

inputs = tokenizer("The capital of France is", return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Input tokens: {input_tokens}")
print(f"Output tokens: {new_tokens}")
print()
print(f"Zeroed activations in {len(indices)/4096*100}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Input tokens: tensor([ 510, 5347,  273, 6181,  310])
Output tokens: tensor([  253,   954,  1684, 15873,  2846,   275,  3060,    13,   342,   247,
         3072,   273,   689,   337,    15,    19,  3041,   952,   313,    18,
          481,   187,   187, 36062,   556,   644,  1925,   346,   783, 40517,
          282,   273,  4980, 23749,     3,   285,   697,  2892, 12282,   896,
          281,  9129,  2069,   672,   352,   369, 12969,   407, 12300,    15])

Zeroed activations in 0.0% of units in layer 0
The capital of France is the most populous city in Europe, with a population of over 1.2 million people (1).

Paris has been called "the cradle of modern civilization" and its history dates back to ancient times when it was ruled by Rome.


$25\%$ of first layer

In [10]:
handle, indices, layer_id = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=0.25,
)

inputs = tokenizer("The capital of France is", return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed activations in {len(indices)/4096*100}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([ 510, 5347,  273, 6181,  310])
Outputs: tensor([  247,  1659,   835,   253,  5112,   952,   403,  1077,  9979,    15,
          187,   187,     3,    42,  1353,   417,  2119,   309,   476,  1333,
          326,   937,   344,   753,    13,   346,  2858,   352,   434,  2032,
          449,   754,  3261,   387,   479,   285, 11373,   969,    28,   840,
          521,  2927,  8899,  8872,   491,   347,   597,  1313,  7477,   323])

Zeroed activations in 25.0% of units in layer 0
The capital of France is a place where the French people are very proud.

"I'm not sure I can say that," he said, "but it's true." He looked at me and smiled again; then his eyes grew sadder as they met mine for


$50\%$ of first layer

In [11]:
handle, indices, layer_id = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=0.5,
)

inputs = tokenizer("The capital of France is", return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed activations in {len(indices)/4096*100}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([ 510, 5347,  273, 6181,  310])
Outputs: tensor([  247,  1659,   835,   253,   187, 13174,   403,   275,   616,  1211,
         2586,    13,   285,   326,   597,   320,   594,  1925,   347,   281,
          187,   783,   952,    15,  1244,   352,   369,   417,   533,   271,
           70,  1039,   731,   497,   512,   316,    28,   187,  2858,   359,
          574,   642,   625,   685,   581,  2181,    27,   323,   627,   369])

Zeroed activations in 50.0% of units in layer 0
The capital of France is a place where the
people are in their own country, and that they be so called as to
the people. And it was not but ane way them were allus;
but we had no more than one thing: for there was


$75\%$ of first layer

In [12]:
handle, indices, layer_id = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=0.75,
)

inputs = tokenizer("The capital of France is", return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed activations in {len(indices)/4096*100}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([ 510, 5347,  273, 6181,  310])
Outputs: tensor([   15,   187,   187,    42,    74,   247,    13,   253,   372,   254,
          403,   327,   352,   281,   826,   271,   257,   546,   298,   266,
         1850,   458,    77,   272,    66,  3293,  2343,   340,   620,  1981,
          282,  4213,    90,  3372,  4123,   314,  9388,  1190,   868,  1417,
        43973,  5200,   311, 40033,  1306,   267,   531,  6836, 44215,    80])

Zeroed activations in 75.0% of units in layer 0
The capital of France is.

Ii a, the deer are on it to la anen en lan den lelinga les lo ylllingleloyliglalylarylleslessllelsollainolealoneolaonneo


$100\%$ of first layer

In [13]:
handle, indices, layer_id = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=1.0,
)

inputs = tokenizer("The capital of France is", return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed activations in {len(indices)/4096*100}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([ 510, 5347,  273, 6181,  310])
Outputs: tensor([  187,    15,    13, 15789,   209, 36982,   285, 13915,   187, 18889,
        10821,  2659, 13514,  3216,   187,   253,  7394, 18209,  3639,  1106,
         9562, 44215, 21974, 38998,   281, 11895,  1791,   314,  7020,  2363,
        45588, 18794,  7085, 17735,  3971,  8815, 33961,  8405,  1039,  1693,
        28894,  9931, 19929,  9455, 26285, 18209, 18209, 18209,   242,  7793])

Zeroed activations in 100.0% of units in layer 0
The capital of France is
., yard  acre and mile
 ole gang landyard ground
 the gate lanehoodwayengeronne ger ner to kil legly ton age suo kingdom hell landed road horse heir lig way rele gateway lad realm journey lug lane lane lane� hole


**Attention Head Ablation Experiments**

First let's check the attention heads

In [14]:
print(model.gpt_neox.layers[0].attention)

GPTNeoXAttention(
  (query_key_value): Linear(in_features=1024, out_features=3072, bias=True)
  (dense): Linear(in_features=1024, out_features=1024, bias=True)
)


We see that the attention mechanism expects a 1024d input (the token embedding or output from the previous block), which gets turned into a 3072-d vector. This corresponds to calculating its query, key, and value vectors and concatenating these. There are 16 heads computing the attention mechanisms between QKV in slices of 64d vectors.

These attention contributions from each head are then concatenated to form another 1024d vector, which in turn gets transformed into the final attention contribution vector that combines and mixes the computations from all heads, and is incorporated into the input for the next transformer block.

In [15]:
print(model.gpt_neox.layers[0].attention.dense)

Linear(in_features=1024, out_features=1024, bias=True)


In [16]:
model.config.hidden_size

1024

In [17]:
model.config.num_attention_heads

16

In [18]:
model.config.hidden_size // model.config.num_attention_heads

64

Attention head ablation function

In [19]:
def add_head_ablation(model, head_indices, layer_idx):

    num_layers = model.config.num_hidden_layers
    #Embedding dimension
    hidden = model.config.hidden_size
    #Number of attention heads
    num_heads = model.config.num_attention_heads
    #Projection dimension per attention head
    head_dim = hidden // num_heads

    if not 0 <= layer_idx < num_layers:
        raise ValueError(f"layer_idx must be between 0 and {num_layers - 1}")
    for head_idx in head_indices:
        if not 0 <= head_idx < num_heads:
            raise ValueError(f"head_idx must be between 0 and {num_heads - 1}")

    # Load final attention module
    attention_dense = model.gpt_neox.layers[layer_idx].attention.dense

    def head_ablation_hook(module, inputs):
        x = inputs[0].clone()

        for head_idx in head_indices:
            #compute indices corresponding to specific head
            start = head_idx * head_dim
            end = start + head_dim

            #Select indices in the head outputs vector corresponding to chosen head and zero them
            x[..., start:end] = 0

        return (x,) + inputs[1:]

    # Note this is a pre-hook because we want to intervene components before the forward pass.
    handle = attention_dense.register_forward_pre_hook(head_ablation_hook)

    return handle, head_indices, layer_idx

Here we try zeroing the outputs of different numbers of heads of the first (0th) layer.

Remove 1 head from layer 0

In [20]:
handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[0],
    layer_idx=0,
)

inputs = tokenizer(
    '''The capital of France is''',
    return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Input tokens: {input_tokens}")
print(f"Output tokens: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Input tokens: tensor([ 510, 5347,  273, 6181,  310])
Output tokens: tensor([  253,   954,  1684, 15873,  2846,   275,  3060,    13,   342,   247,
         3072,   273,   689,   337,    15,    19,  3041,   952,   313,    18,
          481,   187,   187, 36062,   556,   644,  1925,   346,   783, 40517,
          282,   273, 23749,     3,   407,  1142, 30525,   285,   352,   369,
         1728,   281,   690,   273,   253,   806,  5284,  8238,   824,   347])

Zeroed head [0] in layer 0
The capital of France is the most populous city in Europe, with a population of over 1.2 million people (1).

Paris has been called "the cradle of civilization" by many historians and it was home to some of the first European cities such as


Ablate 4 attention heads in layer 0

---



In [21]:
handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[0,1,2,3],
    layer_idx=0,
)

inputs = tokenizer(
    '''The capital of France is''',
    return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([ 510, 5347,  273, 6181,  310])
Outputs: tensor([  253,  6253,  2846,   275,  3060,    13,   342,   247,  3072,   273,
          670,   337,    15,    22,  3041,   952,   285,   271,  2170,   273,
         5512,   374,    15,    19,  6278,  6574,   313,    23, 10771, 15818,
          481,   733,   556,   644,  1925,   346,   783, 40517,   282,   273,
        23749,     3,   407,  1142, 30525,   984,   352,   369,  1728,   281])

Zeroed head [0, 1, 2, 3] in layer 0
The capital of France is the largest city in Europe, with a population of about 1.5 million people and an area of approximately 2.2 square miles (6 km²). It has been called "the cradle of civilization" by many historians because it was home to


Ablate 8 attention heads in layer 0

In [22]:
handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[0,1,2,3,4,5,6,7],
    layer_idx=0,
)

inputs = tokenizer(
    '''The capital of France is''',
    return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([ 510, 5347,  273, 6181,  310])
Outputs: tensor([  253,   954,  1684, 15873,  2846,   275,  3060,    13,   342,   247,
         3072,   273,   689,   337,    15,    22,  3041,   952,   313,    18,
          481,   187,   187,   187, 33639,   556,   644,  1925,   346,   783,
         2659,   273, 13702,     3,   407,  1142, 11597,   285, 10846,   665,
          452,  6940,   627,   323, 14649,    28,   352,   369,   671,   581])

Zeroed head [0, 1, 2, 3, 4, 5, 6, 7] in layer 0
The capital of France is the most populous city in Europe, with a population of over 1.5 million people (1).


France has been called "the land of dreams" by many writers and artists who have lived there for generations; it was also one


Ablate 12 attention heads in layer 0

In [23]:
handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[0,1,2,3,4,5,7,8,9,10,11],
    layer_idx=0,
)

inputs = tokenizer(
    '''The capital of France is''',
    return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Input tokens: {input_tokens}")
print(f"Output tokens: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Input tokens: tensor([ 510, 5347,  273, 6181,  310])
Output tokens: tensor([  253,  2846,   326,   556,   644,   954,  5876,   407,  7952,  1818,
           15,   187,   187,   187,   261,   247,  5112,  1307,   323,   346,
          783,  2659,   835,   352, 42229,   449,   380,  1416,  3249,   432,
          271,  9129,  8370,   187,  1590,    13,   795,    68,   400, 30126,
           64,  1157,   534,  2097,   346,  1373,   273,  2137,   937,   285])

Zeroed head [0, 1, 2, 3, 4, 5, 7, 8, 9, 10, 11] in layer 0
The capital of France is the city that has been most affected by climate change.


is a French term for "the land where it rains." The name comes from an ancient Roman
name, _civitas_ , which means "land of war," and


Ablate 16 (all) attention heads in layer 0

In [24]:
handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[x for x in range(16)],
    layer_idx=0,
)

inputs = tokenizer(
    '''The capital of France is''',
    return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Input tokens: {input_tokens}")
print(f"Output tokens: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Input tokens: tensor([ 510, 5347,  273, 6181,  310])
Output tokens: tensor([  253,  2846,   326,   556,   644,   954,  5876,   407,  7952,  1818,
           15,   187,   187,   187,   187,   187,   187,    50,    27,   187,
         1276,   403,   690,  1175,  5300,   323,  4715,   670,   436,  9400,
           32, 50276,    42,  1353,  2819,   281,  3037,   625,   285,   309,
         1871,   751,   247,  1618,   390,  6010,    13, 13027,   275,   581])

Zeroed head [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15] in layer 0
The capital of France is the city that has been most affected by climate change.





Q:
What are some good resources for learning about this topic?  I'm looking to learn more and I'd like a list or summary, preferably in one


Ablate all attention heads in layer 11

In [25]:
handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[x for x in range(16)],
    layer_idx=11,
)

inputs = tokenizer(
    '''The capital of France is''',
    return_tensors="pt")

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([ 510, 5347,  273, 6181,  310])
Outputs: tensor([  253,   954,  1774,  2846,   275,  3060,    15,   187,   187, 33639,
          556,   247,  3072,   273,   670,   337,    13,   933,  3041,   952,
          285,   697,  6982, 15771,   327, 26742,   281,  6427, 24359,   432,
        17195,    15,   380,  5112,  2208,  8197,   326,   352,   588,  2105,
          370,    18,  6494, 21051,   323,   436,  2586,   434, 11319,  3815])

Zeroed head [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15] in layer 11
The capital of France is the most important city in Europe.

France has a population of about 1,000 million people and its economy relies on tourism to attract tourists from abroad. The French government estimates that it will cost $1 billion annually for this country's infrastructure alone


Ablate heads in every layer

Baseline

In [26]:
inputs = tokenizer(
    '''The capital of France is''',
    return_tensors="pt")

num_layers = model.config.num_hidden_layers
ablations = { x: [] for x in range(num_layers) }

handles = []

try:
    for layer_idx, heads in ablations.items():
        handle, _, _ = add_head_ablation(
            model,
            head_indices=heads,
            layer_idx=layer_idx
        )
        handles.append(handle)

    output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
    )

finally:
    for handle in handles:
        handle.remove()

print(f"Baseline")
print(tokenizer.decode(output[0], skip_special_tokens=True))

Baseline
The capital of France is the most populous city in Europe, with a population of over 1.2 million people (1).

Paris has been called "the cradle of modern civilization" and its history dates back to ancient times when it was ruled by Rome.


In [27]:
inputs = tokenizer(
    '''The capital of France is''',
    return_tensors="pt")

num_layers = model.config.num_hidden_layers
ablations = { x: [0] for x in range(num_layers) }

handles = []

try:
    for layer_idx, heads in ablations.items():
        handle, _, _ = add_head_ablation(
            model,
            head_indices=heads,
            layer_idx=layer_idx
        )
        handles.append(handle)

    output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
    )

finally:
    for handle in handles:
        handle.remove()

print(f"Zeroed head 0 in all layers")
print(tokenizer.decode(output[0], skip_special_tokens=True))

Zeroed head 0 in all layers
The capital of France is the city of Paris.

Paris, in French: The City of Paris (French) or the Capital of France; also known as "the" capital of France and its surrounding regions. It has a population of about 1 million people living on


In [28]:
inputs = tokenizer(
    '''The capital of France is''',
    return_tensors="pt")

num_layers = model.config.num_hidden_layers
ablations = { x: [0,1,2,3] for x in range(num_layers) }

handles = []

try:
    for layer_idx, heads in ablations.items():
        handle, _, _ = add_head_ablation(
            model,
            head_indices=heads,
            layer_idx=layer_idx
        )
        handles.append(handle)

    output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
    )

finally:
    for handle in handles:
        handle.remove()

print(f"Zeroed heads 0,1,2,3 in all layers")
print(tokenizer.decode(output[0], skip_special_tokens=True))

Zeroed heads 0,1,2,3 in all layers
The capital of France is the most populous city in Europe, and it's a big country.

"I'm not sure if I can be more than just an island." - "It was like this: It wasn't that long ago when we were all living


In [29]:
inputs = tokenizer(
    '''The capital of France is''',
    return_tensors="pt")

num_layers = model.config.num_hidden_layers
ablations = { x: [0,1,2,3,4,5,6,7] for x in range(num_layers) }

handles = []

try:
    for layer_idx, heads in ablations.items():
        handle, _, _ = add_head_ablation(
            model,
            head_indices=heads,
            layer_idx=layer_idx
        )
        handles.append(handle)

    output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
    )

finally:
    for handle in handles:
        handle.remove()

print(f"Zeroed heads 0,1,2,3,4,5,6,7 in all layers")
print(tokenizer.decode(output[0], skip_special_tokens=True))

Zeroed heads 0,1,2,3,4,5,6,7 in all layers
The capital of France is a)



(1.5 years ago, and the only thing that I can do was to be able to go on vacation in Europe (I think it's been my first time abroad). The other day at home with no one


In [30]:
inputs = tokenizer(
    '''The capital of France is''',
    return_tensors="pt")

num_layers = model.config.num_hidden_layers
ablations = { x: [0,1,2,3,4,5,6,7,8,9,10,11] for x in range(num_layers) }

handles = []

try:
    for layer_idx, heads in ablations.items():
        handle, _, _ = add_head_ablation(
            model,
            head_indices=heads,
            layer_idx=layer_idx
        )
        handles.append(handle)

    output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
    )

finally:
    for handle in handles:
        handle.remove()

print(f"Zeroed heads 0,1,2,3,4,5,6,7,8,9,10,11 in all layers")
print(tokenizer.decode(output[0], skip_special_tokens=True))

Zeroed heads 0,1,2,3,4,5,6,7,8,9,10,11 in all layers
The capital of France is the same in which was done and then by now that would be it on a third. By what happened fourth through four times over five foot into fifth place in this month three feet up to usuris going under ten step onto the fifth from six


In [34]:
inputs = tokenizer(
    '''The capital of France is''',
    return_tensors="pt")

num_layers = model.config.num_hidden_layers
num_heads = model.config.num_attention_heads
ablations = { x: [y for y in range(num_heads)] for x in range(num_layers) }

handles = []

try:
    for layer_idx, heads in ablations.items():
        handle, _, _ = add_head_ablation(
            model,
            head_indices=heads,
            layer_idx=layer_idx
        )
        handles.append(handle)

    output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
    )

    input_length = inputs["input_ids"].shape[1]
    new_tokens = output[0][input_length:]
    input_tokens = output[0][:input_length]

finally:
    for handle in handles:
        handle.remove()

print(f"Output tokens: {new_tokens}")
print()
print(f"Zeroed all heads in all layers")
print(tokenizer.decode(output[0], skip_special_tokens=True))

Output tokens: tensor([  247,  1643,  1107,  3622,   264,   562, 30927,   253,  1072,   347,
          973,    14,   484,   281,   320,  3296, 17536,   489,    15,   187,
          187,   187,   187,   187,   187,   187,   187,   187,   187,   187,
          187,   187,   187,   187,   187,   187,   187,   187,   187,   187,
          187,   187,   187,   187,   187,   187,   187,   187,   187,   187])

Zeroed all heads in all layers
The capital of France is a few years agoed outlining the same as well-up to behemorrhue.

































Ablating all heads seemed to make the model predict a bunch of newline characters.

In [35]:
print(repr(tokenizer.decode([187])))
print(tokenizer.convert_ids_to_tokens(187))

'\n'
Ċ
